# Load the Train and Validation Data

## Natural-language request

> Create a notebook where you load the train and validation dataset.

This notebook loads `data/splits/train.csv` and `data/splits/val.csv`, written by notebook 08, checks that they are what notebook 08 saved, and separates the **model inputs** from the **label** and the **scoring-only columns**. The tables it leaves ready are the starting point for modelling.

| Name | Contract name | Arrival dates | What it is for |
|---|---|---|---|
| **train** | train | before 2016-07-01 | Fit models, and fit every encoder or scaler. |
| **val** | choose | 2016-07-01 to 2016-12-31 | Compare methods, pick the buffer, enforce the walk ceiling. |

**Test is not loaded.** `test.csv` and `test_labels.csv` stay closed until every choice is frozen (project rule 2).

## Setup

Finds the repository and reads `data/splits/split_columns.json`, which notebook 08 wrote. It records which column has which role and how many rows each part should have, so this notebook does not repeat the column lists by hand.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract

split_dir = repo_root / "data" / "splits"
if not (split_dir / "split_columns.json").exists():
    raise FileNotFoundError("data/splits/ is missing. Run notebooks/08_split_train_val_test.ipynb first.")
manifest = json.loads((split_dir / "split_columns.json").read_text(encoding="utf-8"))

inputs = manifest["inputs"]
provisional = manifest["provisional_inputs"]
label = manifest["label"]
scoring_only = manifest["scoring_only"]

print(f"Inputs ({len(inputs)}): {inputs}")
print(f"  of which provisional, timing not verified ({len(provisional)}): {provisional}")
print(f"Label: {label}")
print(f"Scoring only: {scoring_only}")

Inputs (8): ['hotel', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'agent', 'company']
  of which provisional, timing not verified (4): ['is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'company']
Label: is_canceled
Scoring only: ['arrival_date', 'adr', 'nights']


## Step 1: load the two CSV files

Three reading options keep the values as notebook 08 saved them:

- `index_col="row"`: the first column is the booking's row number in the cleaned table, so every booking can be traced back.
- `parse_dates=["arrival_date"]`: the arrival date comes back as a date, which `contract.nightly_table` needs.
- `dtype={"agent": str, "company": str}`: agent and company IDs such as `9` are labels, not quantities, so they stay text.

In [2]:
def load_split(name):
    return pd.read_csv(split_dir / f"{name}.csv", index_col="row",
                       parse_dates=["arrival_date"], dtype={"agent": str, "company": str})


train = load_split("train")
val = load_split("val")

for name, part in {"train": train, "val": val}.items():
    print(f"{name:>5}: {len(part):,} rows x {part.shape[1]} columns")

train: 49,157 rows x 12 columns
  val: 29,546 rows x 12 columns


## Step 2: check what was loaded

The cell below stops with an error if any of these fails:

1. Each file has the **row count** notebook 08 recorded.
2. Each file has exactly the **columns** notebook 08 recorded: the inputs, the label and the scoring-only columns.
3. Each file's arrival dates fall **inside its contract period**.
4. **No booking is in both** files.
5. The inputs and the label have **no missing values**, and the label is only 0 or 1.

In [3]:
expected_columns = inputs + [label] + scoring_only
periods = {"train": (None, contract.TRAIN_END), "val": (contract.TRAIN_END, contract.PROOF_START)}

for name, part in {"train": train, "val": val}.items():
    assert len(part) == manifest["periods"][name]["rows"], f"{name}: row count differs from notebook 08"
    assert list(part.columns) == expected_columns, f"{name}: columns differ from notebook 08"
    start, end = periods[name]
    assert part["arrival_date"].max() < pd.Timestamp(end), f"{name}: an arrival is after its period"
    if start:
        assert part["arrival_date"].min() >= pd.Timestamp(start), f"{name}: an arrival is before its period"
    assert part[inputs + [label]].notna().all().all(), f"{name}: missing values in inputs or label"
    assert set(part[label].unique()) <= {0, 1}, f"{name}: label is not 0/1"

assert not set(train.index) & set(val.index), "A booking is in both train and val"

print("All five checks pass.")

All five checks pass.


## Step 3: look at the data

The column types, the first rows of train, and a short profile of each part. Categorical inputs are text; the numeric inputs are whole numbers.

In [4]:
display(pd.DataFrame({
    "role": ["provisional input" if column in provisional else "input" for column in inputs]
            + ["label"] + ["scoring only"] * len(scoring_only),
    "type": train.dtypes.astype(str),
}))
display(train.head())

display(pd.DataFrame({
    name: {
        "bookings": f"{len(part):,}",
        "first arrival": f"{part['arrival_date'].min():%Y-%m-%d}",
        "last arrival": f"{part['arrival_date'].max():%Y-%m-%d}",
        "hotel-nights": f"{part.groupby(['hotel', 'arrival_date']).ngroups:,}",
        "cancellation rate": f"{part[label].mean():.1%}",
    }
    for name, part in {"train": train, "val": val}.items()
}))

,role,type
hotel,input,str
market_segment,input,str
distribution_channel,input,str
is_repeated_guest,provisional input,int64
previous_cancellations,provisional input,int64
previous_bookings_not_canceled,provisional input,int64
agent,input,str
company,provisional input,str
is_canceled,label,int64
arrival_date,scoring only,datetime64[us]


,hotel,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,agent,company,is_canceled,arrival_date,adr,nights
row,,,,,,,,,,,,
0,Resort Hotel,Direct,Direct,0,0,0,none,none,0,2015-07-01,0.0,0
1,Resort Hotel,Direct,Direct,0,0,0,none,none,0,2015-07-01,0.0,0
2,Resort Hotel,Direct,Direct,0,0,0,none,none,0,2015-07-01,75.0,1
3,Resort Hotel,Corporate,Corporate,0,0,0,304,none,0,2015-07-01,75.0,1
4,Resort Hotel,Online TA,TA/TO,0,0,0,240,none,0,2015-07-01,98.0,2


,train,val
bookings,"49,157","29,546"
first arrival,2015-07-01,2016-07-01
last arrival,2016-06-30,2016-12-31
hotel-nights,732,368
cancellation rate,35.8%,36.9%


## Step 4: separate inputs, label and scoring columns

- `X_train`, `X_val`: **inputs only**. These are the only columns a model may see.
- `y_train`, `y_val`: the label `is_canceled`.
- `train`, `val`: kept whole. To score a candidate on val, pass `val` and the candidate's probabilities to `contract.nightly_table`, which needs `hotel`, `arrival_date`, `is_canceled`, `adr` and `nights`.

No encoding happens here. Fit any encoder on `X_train` only and apply it unchanged to `X_val`; notebook 08 (Step 5) found `agent` and `company` values in val that never appear in train, so the encoder must accept unknown values.

Four inputs are **provisional** (`is_repeated_guest`, `previous_cancellations`, `previous_bookings_not_canceled`, `company`): used by team decision although their timing is not verified (notebook 06). Every result that uses them must say so.

In [5]:
X_train, y_train = train[inputs], train[label]
X_val, y_val = val[inputs], val[label]

# Hard guard (project rule 3): the label and the scoring-only columns never reach the inputs.
assert label not in X_train.columns and not set(scoring_only) & set(X_train.columns)

print(f"X_train {X_train.shape}, y_train {y_train.shape}")
print(f"X_val   {X_val.shape}, y_val   {y_val.shape}")

X_train (49157, 8), y_train (49157,)
X_val   (29546, 8), y_val   (29546,)


## Summary

- **Train and val are loaded from CSV** and match what notebook 08 saved: same rows, same columns, dates inside the contract periods, no booking in both.
- **Inputs, label and scoring columns are separated**: `X_train`, `y_train`, `X_val`, `y_val`, with `train` and `val` kept whole for scoring.
- **Four of the eight inputs are provisional**; their timing is not verified.
- **Test is not loaded.**

**Next:** fit the flat-rule baseline and the candidate methods on train, score them on val with `contract.nightly_table` and `contract.night_cost`, and report euros, empty rooms and walked guests against the walk ceiling (project rules 5 and 6).